# Micro-organism Image Classification — Vision Transformer (From Scratch, and Pretrained)

**Goal:** Same micro-organism dataset as every other notebook in this project, but a genuinely
different kind of architecture: the **Vision Transformer (ViT)**. Every model built so far —
AlexNet, VGG16, GoogLeNet, MobileNetV2, EfficientNetB0, ResNet50 — is fundamentally a stack of
**convolutions**. ViT (Dosovitskiy et al., 2020, *"An Image is Worth 16x16 Words"*) throws that out:
it slices an image into a grid of patches and feeds them through the same **Transformer** design
originally built for text (Vaswani et al., 2017, *"Attention Is All You Need"*).

**This notebook covers two things, back to back, matching the two styles already used elsewhere in
this project:**

- **Part A — Built From Scratch.** A small ("mini") ViT, hand-built layer by layer, the same way
  AlexNet/VGG16/GoogLeNet were — so you can see exactly how patch embeddings, position embeddings,
  and self-attention actually work.
- **Part B — Pretrained, Fine-Tuned.** A real, full-size ViT-Base/16, pretrained on millions of
  images, fine-tuned the same two-phase way as MobileNetV2/EfficientNetB0/ResNet50.

**An honest heads-up before we start:** the ViT paper itself found that Vision Transformers need
*far* more training data than CNNs to reach their potential, because they lack a CNN's "built-in"
assumptions about images (that nearby pixels are related, that a pattern means the same thing
wherever it appears). With only 633 training images, **expect Part A to underperform every CNN
built earlier in this project** — that's not a bug, it's the exact tradeoff the paper documents, and
exactly why Part B exists: pretraining is how real-world ViT usage gets around this weakness.

**One new dependency:** `keras.applications` doesn't include a Vision Transformer, so Part B uses
**KerasHub** — the Keras team's own library of additional pretrained models — installed in Step 1.

## Step 1: Import the Libraries (and Install KerasHub)

Most of these imports are needed for Part A (the from-scratch build): `LayerNormalization`,
`MultiHeadAttention`, and `Add` are the three building blocks a Transformer needs that no earlier
notebook in this project has used before.

`keras_hub` is new — it's only needed for Part B, to load a pretrained ViT. It doesn't ship with
TensorFlow by default, so the first cell below installs it.

In [ ]:
!pip install --quiet keras-hub

In [ ]:
# Basic Python tools for handling files and folders
import os
from pathlib import Path

# Numerical operations
import numpy as np

# For plotting images and graphs
import matplotlib.pyplot as plt

# Deep learning library
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input, Dense, Dropout, Layer, Embedding,
    LayerNormalization, MultiHeadAttention, Add, GlobalAveragePooling1D, Lambda
)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau

# Used to compute class weights for imbalanced datasets (Part B only)
from sklearn.utils.class_weight import compute_class_weight

# KerasHub -- only used in Part B, to load a pretrained Vision Transformer
import keras_hub

print("TensorFlow version:", tf.__version__)
print("KerasHub version:", keras_hub.__version__)

## Step 2: Set the Dataset Path and Basic Parameters

All the settings for both parts of this notebook, grouped together and commented by which part uses
them.

- `IMG_HEIGHT`, `IMG_WIDTH` — 224x224, same as the rest of this project.
- `PATCH_SIZE` — Part A's from-scratch ViT splits each image into non-overlapping 16x16 patches
  (matching the "16x16 Words" in the paper's own title, and the pretrained model's own patch size
  in Part B, so the two stay comparable).
- `PROJECTION_DIM`, `NUM_HEADS`, `TRANSFORMER_LAYERS`, `MLP_DIM` — Part A's *mini* ViT size. The real
  ViT-Base/16 used in Part B uses 768 / 12 / 12 / 3072 for these same four settings — dramatically
  bigger, which is exactly why it needs pretraining and Part A's small version doesn't stand a
  chance of matching it from scratch on this small dataset.
- `EPOCHS_SCRATCH` — training budget for Part A.
- `PHASE1_EPOCHS`, `PHASE2_EPOCHS`, `FINE_TUNE_BLOCKS` — Part B's two-phase fine-tuning budget.
  ViT-Base/16 has 12 Transformer blocks in total; `FINE_TUNE_BLOCKS = 2` unfreezes just the last 2
  of them during Phase 2.

**Important:** Update `DATASET_DIR` below if your folder path is different.

In [ ]:
# Path to the main dataset folder (one sub-folder per micro-organism type).
DATASET_DIR = Path(r"/home/asus/projects/computer_vision/Micro-organism Classificationn/Micro_Organism")

# Shared by both parts
IMG_HEIGHT = 224
IMG_WIDTH = 224
IMG_SIZE = (IMG_HEIGHT, IMG_WIDTH)
BATCH_SIZE = 32
VALIDATION_SPLIT = 0.2
SEED = 42

# --- Part A: from-scratch "mini" ViT settings ---
PATCH_SIZE = 16
NUM_PATCHES = (IMG_HEIGHT // PATCH_SIZE) ** 2   # 14 x 14 = 196 patches
PROJECTION_DIM = 64      # embedding size for each patch (ViT-Base/16 uses 768)
NUM_HEADS = 4            # attention heads per Transformer block (ViT-Base/16 uses 12)
TRANSFORMER_LAYERS = 4   # number of stacked Transformer blocks (ViT-Base/16 uses 12)
MLP_DIM = 128            # hidden size inside each block's MLP (ViT-Base/16 uses 3072)
EPOCHS_SCRATCH = 40

# --- Part B: pretrained ViT-Base/16 settings ---
PHASE1_EPOCHS = 10
PHASE2_EPOCHS = 15
FINE_TUNE_BLOCKS = 2     # out of ViT-Base/16's 12 Transformer blocks

## Step 3: Explore the Dataset

Same exploration as every other notebook in this project.

In [ ]:
class_names = sorted([folder.name for folder in DATASET_DIR.iterdir() if folder.is_dir()])

print("Number of micro-organism types (classes) found:", len(class_names))
for name in class_names:
    print(" -", name)

In [ ]:
image_counts = {}

for class_name in class_names:
    class_folder = DATASET_DIR / class_name
    num_images = len([f for f in class_folder.iterdir() if f.suffix.lower() in [".jpg", ".jpeg", ".png"]])
    image_counts[class_name] = num_images

print("Number of images per micro-organism type:")
for class_name, count in image_counts.items():
    print(f" - {class_name}: {count} images")

print("\nTotal images:", sum(image_counts.values()))

plt.figure(figsize=(10, 5))
plt.bar(image_counts.keys(), image_counts.values(), color="mediumpurple")
plt.xticks(rotation=45, ha="right")
plt.xlabel("Micro-organism Type")
plt.ylabel("Number of Images")
plt.title("Number of Images per Micro-organism Type")
plt.tight_layout()
plt.show()

## Step 4: Visualize Sample Images

A quick visual check that the images loaded correctly.

In [ ]:
plt.figure(figsize=(15, 10))

for i, class_name in enumerate(class_names):
    class_folder = DATASET_DIR / class_name

    image_files = [f for f in class_folder.iterdir() if f.suffix.lower() in [".jpg", ".jpeg", ".png"]]
    if len(image_files) == 0:
        continue
    sample_image_path = image_files[0]

    img = keras.preprocessing.image.load_img(sample_image_path)
    plt.subplot(3, 4, i + 1)  # Adjust grid size (rows, cols) if you have more/fewer classes
    plt.imshow(img)
    plt.title(class_name, fontsize=10)
    plt.axis("off")

plt.tight_layout()
plt.show()

---
# Part A: Vision Transformer, Built From Scratch
---

Same "no pretrained weights" spirit as the AlexNet/VGG16/GoogLeNet notebook — everything below is
built and trained from a random starting point.

## Step 5: Prepare the Data for Training (Part A)

Built from scratch, so — like AlexNet/VGG16/GoogLeNet — just a plain `rescale=1./255` to bring pixel
values into `[0, 1]`, plus the same data augmentation used throughout this project's from-scratch
notebooks.

In [ ]:
train_datagen_scratch = ImageDataGenerator(
    rescale=1.0 / 255,
    rotation_range=20,
    width_shift_range=0.1,
    height_shift_range=0.1,
    zoom_range=0.1,
    brightness_range=(0.8, 1.2),
    shear_range=0.1,
    horizontal_flip=True,
    validation_split=VALIDATION_SPLIT
)

val_datagen_scratch = ImageDataGenerator(
    rescale=1.0 / 255,
    validation_split=VALIDATION_SPLIT
)

In [ ]:
train_generator_scratch = train_datagen_scratch.flow_from_directory(
    DATASET_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    subset="training",
    seed=SEED
)

validation_generator_scratch = val_datagen_scratch.flow_from_directory(
    DATASET_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    subset="validation",
    seed=SEED
)

num_classes = len(class_names)
print("Class indices used by the model:", train_generator_scratch.class_indices)

## Step 6: Theory — Inside the Vision Transformer

1. **Treat an image as a sentence of patches.** A Transformer expects a *sequence* of tokens (like
   words in a sentence). ViT builds that sequence by slicing the image into a grid of fixed-size,
   non-overlapping patches — 16x16 pixels each, here — and treats **each patch as if it were a single
   "word."** A 224x224 image cut into 16x16 patches becomes a sequence of `(224/16)^2 = 196` "words,"
   which is exactly what the `Patches` layer below computes.

2. **Patch + position embedding.** Each 16x16x3 patch is flattened into one 768-length vector of raw
   pixel values (`16 x 16 x 3`), then passed through a single shared `Dense` layer — a linear
   projection — that turns it into a `PROJECTION_DIM`-length embedding, exactly like a word-embedding
   layer turns a word ID into a dense vector in NLP. Since self-attention (next) has no built-in
   sense of order, a **learned position embedding** is added to every patch embedding too, so the
   network can tell "this patch was top-left" apart from "this patch was bottom-right."

3. **Self-attention: every patch looks directly at every other patch.** This is the biggest
   conceptual difference from every convolutional network in this project. A convolution can only
   combine information from *nearby* pixels — a single 3x3 filter's receptive field starts tiny and
   only grows gradually as more layers stack up (the VGG notebook spent real space on exactly this).
   Self-attention has no such restriction: in a *single* attention layer, patch #1 can directly attend
   to patch #196, no matter "how far apart" they are. Concretely, `MultiHeadAttention` computes, for
   every patch, a **Query**, **Key**, and **Value** vector; each patch's Query is compared against
   every *other* patch's Key to produce attention scores ("how much should I pay attention to you?"),
   a softmax turns those scores into weights, and each patch's new representation becomes a weighted
   blend of every patch's Value vector. "Multi-head" just means this whole process runs several times
   in parallel (`NUM_HEADS = 4` here) with separately learned projections, so the model can track
   several different kinds of relationships between patches at once.

4. **The Transformer encoder block.** Each of the `TRANSFORMER_LAYERS` blocks stacked in this network
   repeats the same two-part pattern, each part wrapped in a residual (skip) connection — the exact
   same "add the block's input back to its output" idea from ResNet, reused here in a completely
   different kind of architecture:
   - `LayerNormalization` -> Multi-Head Self-Attention -> add the block's input back (residual #1)
   - `LayerNormalization` -> a small 2-layer MLP (using GELU, a smoother cousin of ReLU) -> add the
     previous result back (residual #2)

5. **Getting to a single prediction.** The original paper prepends a special, learnable "class token"
   to the sequence of patches (an idea borrowed from BERT), and uses only *its* final representation
   for classification. For simplicity — and to stay consistent with the `GlobalAveragePooling2D`-based
   heads used everywhere else in this project — this notebook instead averages the final
   representations of all 196 patches together (`GlobalAveragePooling1D`) before the classification
   layers. Both approaches are used in real ViT implementations; averaging is a little simpler to
   implement and works comparably well in practice.

6. **Why two small custom layers, `Patches` and `PatchEncoder`?** `Patches` does the
   slicing-into-a-grid step from idea #1 (using `tf.image.extract_patches`); `PatchEncoder` does the
   linear projection + position embedding from idea #2. Splitting these into two focused layers,
   instead of one giant function, follows the same reasoning behind this project's `conv_block` /
   `inception_module` helpers in the GoogLeNet notebook.

**Shape, from a 224x224x3 input — verified by actually building and running this exact model:**

| Stage | What happens | Output shape |
|---|---|---|
| Input | — | 224x224x3 |
| `Patches` | Slice into a 14x14 grid of 16x16 patches, flatten each | 196 patches x 768 |
| `PatchEncoder` | Linear projection + learned position embedding | 196 x `PROJECTION_DIM` (64) |
| Transformer blocks x4 | Self-attention + MLP, each wrapped in a residual connection | 196 x 64 (shape unchanged) |
| Final `LayerNormalization` + `GlobalAveragePooling1D` | Average all 196 patch representations into one vector | 64 |
| Classification head | `Dense(128, gelu)` -> `Dropout` -> `Dense(softmax)` | `num_classes` |

**Why this from-scratch version is deliberately tiny:** the real ViT-Base/16 (used pretrained in
Part B) uses `PROJECTION_DIM=768`, `NUM_HEADS=12`, `TRANSFORMER_LAYERS=12`, `MLP_DIM=3072` — around
200x more parameters than the mini version below. Building that full size from scratch on 633
training images would overfit almost immediately; this smaller version is sized to at least have a
fighting chance, while still showing every piece of real ViT theory in action.

In [ ]:
class Patches(Layer):
    """Slices a batch of images into a grid of non-overlapping patch_size x patch_size patches,
    and flattens each patch into a single vector (see Step 6, idea #1).
    """
    def __init__(self, patch_size, **kwargs):
        super().__init__(**kwargs)
        self.patch_size = patch_size

    def call(self, images):
        batch_size = tf.shape(images)[0]
        patches = tf.image.extract_patches(
            images=images,
            sizes=[1, self.patch_size, self.patch_size, 1],
            strides=[1, self.patch_size, self.patch_size, 1],
            rates=[1, 1, 1, 1],
            padding="VALID"
        )
        patch_dims = patches.shape[-1]
        patches = tf.reshape(patches, [batch_size, -1, patch_dims])
        return patches


class PatchEncoder(Layer):
    """Linearly projects each flattened patch into `projection_dim`, then adds a learned position
    embedding so the model can tell patches apart by where they came from (see Step 6, idea #2).
    """
    def __init__(self, num_patches, projection_dim, **kwargs):
        super().__init__(**kwargs)
        self.num_patches = num_patches
        self.projection = Dense(units=projection_dim)
        self.position_embedding = Embedding(input_dim=num_patches, output_dim=projection_dim)

    def call(self, patch):
        positions = tf.range(start=0, limit=self.num_patches, delta=1)
        encoded = self.projection(patch) + self.position_embedding(positions)
        return encoded


def transformer_block(x, projection_dim, num_heads, mlp_dim, dropout_rate=0.1):
    """One Transformer encoder block: Multi-Head Self-Attention, then an MLP, each wrapped in its
    own LayerNormalization + residual connection (see Step 6, idea #4).
    """
    # --- Self-attention sub-block ---
    x1 = LayerNormalization(epsilon=1e-6)(x)
    attention_output = MultiHeadAttention(
        num_heads=num_heads, key_dim=projection_dim, dropout=dropout_rate
    )(x1, x1)
    x2 = Add()([attention_output, x])

    # --- MLP sub-block ---
    x3 = LayerNormalization(epsilon=1e-6)(x2)
    mlp_output = Dense(mlp_dim, activation="gelu")(x3)
    mlp_output = Dropout(dropout_rate)(mlp_output)
    mlp_output = Dense(projection_dim)(mlp_output)
    mlp_output = Dropout(dropout_rate)(mlp_output)
    x4 = Add()([mlp_output, x2])

    return x4


def build_vit_scratch(input_shape, num_classes):
    """Builds the small ("mini") Vision Transformer used in Part A."""
    inputs = Input(shape=input_shape)

    patches = Patches(PATCH_SIZE)(inputs)
    encoded_patches = PatchEncoder(NUM_PATCHES, PROJECTION_DIM)(patches)

    for _ in range(TRANSFORMER_LAYERS):
        encoded_patches = transformer_block(encoded_patches, PROJECTION_DIM, NUM_HEADS, MLP_DIM)

    representation = LayerNormalization(epsilon=1e-6)(encoded_patches)
    representation = GlobalAveragePooling1D()(representation)
    representation = Dropout(0.3)(representation)

    features = Dense(128, activation="gelu")(representation)
    features = Dropout(0.3)(features)
    outputs = Dense(num_classes, activation="softmax")(features)

    model = Model(inputs, outputs, name="ViT_scratch")
    return model


vit_scratch_model = build_vit_scratch(input_shape=(IMG_HEIGHT, IMG_WIDTH, 3), num_classes=num_classes)

## Step 7: Compile the From-Scratch ViT

Same optimizer style as the other from-scratch notebooks. Transformers are commonly trained with a
slightly lower learning rate than CNNs, since attention layers can be less stable early in training.

In [ ]:
vit_scratch_model.compile(
    optimizer=Adam(learning_rate=5e-5),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

vit_scratch_model.summary()

## Step 8: Set Up Training Callbacks

In [ ]:
early_stop_vit_scratch = EarlyStopping(
    monitor="val_loss",
    patience=6,
    restore_best_weights=True
)

checkpoint_vit_scratch = ModelCheckpoint(
    "best_microorganism_vit_scratch.keras",
    monitor="val_accuracy",
    save_best_only=True
)

reduce_lr_vit_scratch = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=3,
    min_lr=1e-7
)

## Step 9: Train the From-Scratch ViT

As flagged at the top of this notebook, don't expect this to beat the CNNs — a Transformer with no
convolutional "head start" on a dataset this small is exactly the scenario the ViT paper warns about.

In [ ]:
history_vit_scratch = vit_scratch_model.fit(
    train_generator_scratch,
    validation_data=validation_generator_scratch,
    epochs=EPOCHS_SCRATCH,
    callbacks=[early_stop_vit_scratch, checkpoint_vit_scratch, reduce_lr_vit_scratch]
)

## Step 10: Plot the From-Scratch ViT's Training History

In [ ]:
plt.figure(figsize=(14, 5))

plt.subplot(1, 2, 1)
plt.plot(history_vit_scratch.history["accuracy"], label="Training Accuracy")
plt.plot(history_vit_scratch.history["val_accuracy"], label="Validation Accuracy")
plt.title("ViT (Scratch): Accuracy over Epochs")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history_vit_scratch.history["loss"], label="Training Loss")
plt.plot(history_vit_scratch.history["val_loss"], label="Validation Loss")
plt.title("ViT (Scratch): Loss over Epochs")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()

plt.tight_layout()
plt.show()

## Step 11: Evaluate the From-Scratch ViT

In [ ]:
vit_scratch_val_loss, vit_scratch_val_accuracy = vit_scratch_model.evaluate(validation_generator_scratch)

print(f"ViT (Scratch) Validation Loss: {vit_scratch_val_loss:.4f}")
print(f"ViT (Scratch) Validation Accuracy: {vit_scratch_val_accuracy:.4f} ({vit_scratch_val_accuracy * 100:.2f}%)")

## Step 12: Test the From-Scratch ViT on Some Sample Images

In [ ]:
sample_images, sample_labels = next(validation_generator_scratch)
predictions = vit_scratch_model.predict(sample_images)

index_to_class = {v: k for k, v in train_generator_scratch.class_indices.items()}

plt.figure(figsize=(16, 8))
num_to_show = min(8, len(sample_images))

for i in range(num_to_show):
    plt.subplot(2, 4, i + 1)
    plt.imshow(sample_images[i])

    predicted_class = index_to_class[np.argmax(predictions[i])]
    true_class = index_to_class[np.argmax(sample_labels[i])]
    confidence = np.max(predictions[i]) * 100

    color = "green" if predicted_class == true_class else "red"
    plt.title(f"Pred: {predicted_class}\nTrue: {true_class}\n({confidence:.1f}%)",
              color=color, fontsize=9)
    plt.axis("off")

plt.tight_layout()
plt.show()

## Step 13: Save the From-Scratch ViT

In [ ]:
vit_scratch_model.save("microorganism_vit_scratch.keras")
print("Model saved as microorganism_vit_scratch.keras")

---
# Part B: Pretrained Vision Transformer (Fine-Tuned)
---

Same two-phase transfer-learning workflow as the MobileNetV2/EfficientNetB0/ResNet50 notebooks —
now applied to a real, full-size **ViT-Base/16**, pretrained on millions of images.

## Step 14: Prepare the Data for Training (Part B)

ViT was pretrained with images scaled to **`[-1, 1]`** (each channel: divide by 255, then subtract
0.5 and divide by 0.5 again) — the exact same scaling `MobileNetV2` uses elsewhere in this project,
and different from both EfficientNet's and ResNet50's own preprocessing. Since KerasHub doesn't
provide a standalone `preprocess_input` function the way `keras.applications` does, we write the
(very short) equivalent ourselves.

In [ ]:
def vit_preprocess_input(x):
    # Matches the pretrained ViT-Base/16's expected input: scale to [-1, 1]
    return (x / 127.5) - 1.0

In [ ]:
train_datagen_pretrained = ImageDataGenerator(
    preprocessing_function=vit_preprocess_input,
    rotation_range=20,
    width_shift_range=0.1,
    height_shift_range=0.1,
    zoom_range=0.1,
    horizontal_flip=True,
    validation_split=VALIDATION_SPLIT
)

val_datagen_pretrained = ImageDataGenerator(
    preprocessing_function=vit_preprocess_input,
    validation_split=VALIDATION_SPLIT
)

In [ ]:
train_generator_pretrained = train_datagen_pretrained.flow_from_directory(
    DATASET_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    subset="training",
    seed=SEED
)

validation_generator_pretrained = val_datagen_pretrained.flow_from_directory(
    DATASET_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    subset="validation",
    seed=SEED
)

print("Class indices used by the model:", train_generator_pretrained.class_indices)

## Step 15: Compute Class Weights

Same reasoning as the MobileNetV2/EfficientNetB0/ResNet50 notebooks: give rarer micro-organism types
a higher weight so the model can't get a misleadingly good score just by favoring common types.

In [ ]:
class_labels = np.unique(train_generator_pretrained.classes)

class_weights_array = compute_class_weight(
    class_weight="balanced",
    classes=class_labels,
    y=train_generator_pretrained.classes
)

class_weights = dict(zip(class_labels, class_weights_array))

print("Class weights (higher = fewer training images for that type):")
for class_index, weight in class_weights.items():
    type_name = [name for name, idx in train_generator_pretrained.class_indices.items() if idx == class_index][0]
    print(f" - {type_name}: {weight:.2f}")

## Step 16: Theory — What's Different About the Pretrained ViT

Everything from Step 6 (patches, position embeddings, self-attention, Transformer blocks) still
applies exactly as described — Part B uses the *same architecture family*, just built at *real*
size and with weights already learned from data, instead of a small version starting from random
numbers.

1. **Real size: ViT-Base/16.** Where Part A used `PROJECTION_DIM=64`, `NUM_HEADS=4`,
   `TRANSFORMER_LAYERS=4`, `MLP_DIM=128`, the pretrained checkpoint used here — Google's
   `vit_base_patch16_224`, via KerasHub — uses `768 / 12 / 12 / 3072` for those same four settings:
   **~86 million parameters**, roughly 200x more than Part A's mini version.

2. **Why pretraining matters *even more* for ViT than it did for the CNNs.** Step 6 already flagged
   that Transformers lack a CNN's built-in assumptions about images, so they need far more data to
   learn good features from scratch. Pretraining is precisely how that weakness gets neutralized:
   this checkpoint was pretrained on ImageNet-21k (about 14 million images across 21,000 categories)
   and then fine-tuned on ImageNet-1k — so by the time it reaches our 633 microscope images, it has
   already learned what edges, textures, shapes, and objects generally look like, the hard way, from
   real data, instead of needing our tiny dataset to teach it that from nothing.

3. **Why KerasHub instead of `keras.applications`?** `keras.applications` (used for MobileNetV2,
   EfficientNetB0, and ResNet50 earlier) doesn't include a Vision Transformer. **KerasHub** is the
   Keras team's own separate library for additional pretrained models, and — helpfully — it follows
   the same basic idea: `keras_hub.models.ViTBackbone.from_preset(...)` plays the same role as
   `ResNet50(weights="imagenet", include_top=False)` did before: load a pretrained feature extractor,
   without its original classification head, ready to attach a new one.

4. **Getting a classification-ready vector out.** Part A averaged all 196 patch representations
   together (`GlobalAveragePooling1D`). The pretrained backbone instead outputs a sequence shaped
   `(197, 768)` — 196 patches *plus* the original paper's class token at position 0 — so Step 17 below
   picks out just that first position (`output[:, 0, :]`) as the image's overall representation,
   matching how this specific pretrained checkpoint was actually trained to be used.

## Step 17: Build the Model — Pretrained ViT Backbone + Classification Head

`from_preset(..., load_weights=True)` downloads and loads the pretrained ViT-Base/16 weights. Just
like the CNN backbones, we then freeze the whole thing for Phase 1, and add our own small
classification head on top of the class-token output described in Step 16, idea #4.

In [ ]:
base_model = keras_hub.models.ViTBackbone.from_preset(
    "vit_base_patch16_224_imagenet",
    load_weights=True
)

# Freeze the entire pretrained backbone for Phase 1
base_model.trainable = False

print("Number of Transformer blocks in the base model:", len(base_model.get_layer("vit_encoder").encoder_layers))

In [ ]:
# base_model.output has shape (batch, 197, 768): 196 patch tokens + 1 class token, each 768-long.
# We pick out just the class token (position 0) as the image's overall representation --
# see Step 16, idea #4.
sequence_output = base_model.output
cls_token = Lambda(lambda t: t[:, 0, :])(sequence_output)

x = Dense(128, activation="relu")(cls_token)
x = Dropout(0.3)(x)

num_classes = len(class_names)
predictions = Dense(num_classes, activation="softmax")(x)

vit_pretrained_model = Model(inputs=base_model.input, outputs=predictions)

## Step 18: Phase 1 — Train the Head With the Base Model Frozen

In [ ]:
vit_pretrained_model.compile(
    optimizer=Adam(learning_rate=1e-3),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

vit_pretrained_model.summary()

In [ ]:
early_stop_phase1 = EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True
)

checkpoint_phase1 = ModelCheckpoint(
    "best_microorganism_vit_pretrained_phase1.keras",
    monitor="val_accuracy",
    save_best_only=True
)

history_phase1 = vit_pretrained_model.fit(
    train_generator_pretrained,
    validation_data=validation_generator_pretrained,
    epochs=PHASE1_EPOCHS,
    class_weight=class_weights,
    callbacks=[early_stop_phase1, checkpoint_phase1]
)

## Step 19: Phase 2 — Fine-Tune the Last Few Transformer Blocks

Same "unfreeze a little, update very gently" idea as the CNN notebooks, adapted to a Transformer's
block-based structure: instead of counting individual layers, we unfreeze whole Transformer blocks.

**A ViT-specific detail worth knowing:** because each Transformer block is itself made of several
nested pieces (attention, its own normalization layers, the MLP), simply setting
`base_model.trainable = True` and then trying to re-freeze most of it doesn't reliably work the way
it did for the CNN backbones. Instead, we reset the whole backbone to trainable first, then
explicitly freeze the patch/position embedding and every block *except* the last `FINE_TUNE_BLOCKS`.

In [ ]:
# Reset, then re-freeze everything except the last FINE_TUNE_BLOCKS Transformer blocks
base_model.trainable = True

patch_embedding = base_model.get_layer("vit_patching_and_embedding")
encoder = base_model.get_layer("vit_encoder")

patch_embedding.trainable = False

for block in encoder.encoder_layers[:-FINE_TUNE_BLOCKS]:
    block.trainable = False
for block in encoder.encoder_layers[-FINE_TUNE_BLOCKS:]:
    block.trainable = True

print(f"Unfrozen Transformer blocks: {FINE_TUNE_BLOCKS} out of {len(encoder.encoder_layers)}")

In [ ]:
# Re-compile with a much smaller learning rate now that part of the pretrained backbone can update
vit_pretrained_model.compile(
    optimizer=Adam(learning_rate=1e-5),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

vit_pretrained_model.summary()

In [ ]:
early_stop_phase2 = EarlyStopping(
    monitor="val_loss",
    patience=4,
    restore_best_weights=True
)

checkpoint_phase2 = ModelCheckpoint(
    "best_microorganism_vit_pretrained_finetuned.keras",
    monitor="val_accuracy",
    save_best_only=True
)

reduce_lr = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=2,
    min_lr=1e-7
)

history_phase2 = vit_pretrained_model.fit(
    train_generator_pretrained,
    validation_data=validation_generator_pretrained,
    epochs=PHASE2_EPOCHS,
    class_weight=class_weights,
    callbacks=[early_stop_phase2, checkpoint_phase2, reduce_lr]
)

## Step 20: Plot the Full Training History (Both Phases Combined)

In [ ]:
acc = history_phase1.history["accuracy"] + history_phase2.history["accuracy"]
val_acc = history_phase1.history["val_accuracy"] + history_phase2.history["val_accuracy"]
loss = history_phase1.history["loss"] + history_phase2.history["loss"]
val_loss = history_phase1.history["val_loss"] + history_phase2.history["val_loss"]

phase1_len = len(history_phase1.history["accuracy"])

plt.figure(figsize=(14, 5))

plt.subplot(1, 2, 1)
plt.plot(acc, label="Training Accuracy")
plt.plot(val_acc, label="Validation Accuracy")
plt.axvline(x=phase1_len - 1, color="gray", linestyle="--", label="Fine-tuning starts")
plt.title("ViT (Pretrained): Accuracy over Epochs")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(loss, label="Training Loss")
plt.plot(val_loss, label="Validation Loss")
plt.axvline(x=phase1_len - 1, color="gray", linestyle="--", label="Fine-tuning starts")
plt.title("ViT (Pretrained): Loss over Epochs")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()

plt.tight_layout()
plt.show()

## Step 21: Evaluate the Pretrained ViT

In [ ]:
vit_pretrained_val_loss, vit_pretrained_val_accuracy = vit_pretrained_model.evaluate(validation_generator_pretrained)

print(f"ViT (Pretrained) Validation Loss: {vit_pretrained_val_loss:.4f}")
print(f"ViT (Pretrained) Validation Accuracy: {vit_pretrained_val_accuracy:.4f} ({vit_pretrained_val_accuracy * 100:.2f}%)")

## Step 22: Test the Pretrained ViT on Some Sample Images

ViT's preprocessing only rescales pixel values (no channel swap, unlike ResNet50) — so undoing it
for display just needs the rescale reversed, not a channel-order fix.

In [ ]:
sample_images, sample_labels = next(validation_generator_pretrained)
predictions = vit_pretrained_model.predict(sample_images)

index_to_class = {v: k for k, v in train_generator_pretrained.class_indices.items()}

plt.figure(figsize=(16, 8))
num_to_show = min(8, len(sample_images))

for i in range(num_to_show):
    plt.subplot(2, 4, i + 1)

    # Undo vit_preprocess_input's [-1, 1] scaling back to a displayable [0, 1] range
    img_display = (sample_images[i] + 1.0) / 2.0
    img_display = np.clip(img_display, 0, 1)
    plt.imshow(img_display)

    predicted_class = index_to_class[np.argmax(predictions[i])]
    true_class = index_to_class[np.argmax(sample_labels[i])]
    confidence = np.max(predictions[i]) * 100

    color = "green" if predicted_class == true_class else "red"
    plt.title(f"Pred: {predicted_class}\nTrue: {true_class}\n({confidence:.1f}%)",
              color=color, fontsize=9)
    plt.axis("off")

plt.tight_layout()
plt.show()

## Step 23: Save the Final Model

In [ ]:
vit_pretrained_model.save("microorganism_vit_pretrained_finetuned.keras")
print("Model saved as microorganism_vit_pretrained_finetuned.keras")

## Comparing Part A and Part B

| | ViT (Scratch, Part A) | ViT (Pretrained, Part B) |
|---|---|---|
| Parameters | ~400 thousand (mini config) | ~86 million (ViT-Base/16) |
| Pretrained? | No — random initialization | Yes — ImageNet-21k, then ImageNet-1k |
| Transformer blocks | 4 | 12 (last 2 fine-tuned) |
| Patch embedding size | 64 | 768 |

**The big picture, across every notebook in this project so far:**

| | Built from scratch | Pretrained, fine-tuned |
|---|---|---|
| Convolutional | AlexNet, VGG16, GoogLeNet | MobileNetV2, EfficientNetB0, ResNet50 |
| Transformer | ViT (Part A) | ViT (Part B) |

The pattern that should show up in the numbers below is arguably the clearest lesson of this entire
project: **architecture choice matters, but for a dataset this small (789 images), whether a model
was pretrained matters more.** Every "built from scratch" model here had to learn everything —
including basic ideas like "edges exist" and "nearby pixels are usually related" — from just 633
training images. Every pretrained model already knew those things before it ever saw a single
micro-organism.

## Next Steps / Ideas to Improve Further (Optional)

- Try a bigger "mini" ViT in Part A (e.g. `PROJECTION_DIM=128`, `TRANSFORMER_LAYERS=6`) to see how
  much of the gap to Part B closes with size alone, with no pretraining at all.
- Try `FINE_TUNE_BLOCKS = 4` in Part B to unfreeze more of the pretrained backbone.
- Visualize attention: KerasHub's `ViTBackbone` can be adapted to also return attention weights,
  which can be plotted as a heatmap over the original image to see which patches the model is
  actually "looking at" when it makes a decision.
- Build a confusion matrix for both parts and compare which micro-organism types each one confuses —
  does the from-scratch ViT make the "same kinds" of mistakes as the from-scratch CNNs, or different
  ones?

In [ ]:
from IPython.display import display, Markdown

# Known results from the earlier notebooks in this project -- see those notebooks for details
alexnet_val_loss, alexnet_val_accuracy = 1.7707, 0.2949
vgg_val_loss, vgg_val_accuracy = 1.8873, 0.2821
mobilenet_val_loss, mobilenet_val_accuracy = 1.0590, 0.6667
efficientnet_val_loss, efficientnet_val_accuracy = 1.0385, 0.6667

# Both ViT results come straight from Step 11 and Step 21 (Evaluate), computed earlier in this notebook

comparison_text = f"""
## Final Validation Comparison — Every Model With a Known Result

| Model | Type | Validation Loss | Validation Accuracy |
|---|---|---:|---:|
| AlexNet (scratch) | CNN, from scratch | {alexnet_val_loss:.4f} | {alexnet_val_accuracy * 100:.2f}% |
| VGG16 (scratch) | CNN, from scratch | {vgg_val_loss:.4f} | {vgg_val_accuracy * 100:.2f}% |
| MobileNetV2 (frozen) | CNN, pretrained | {mobilenet_val_loss:.4f} | {mobilenet_val_accuracy * 100:.2f}% |
| EfficientNetB0 (fine-tuned) | CNN, pretrained | {efficientnet_val_loss:.4f} | {efficientnet_val_accuracy * 100:.2f}% |
| ViT (scratch, this notebook) | Transformer, from scratch | {vit_scratch_val_loss:.4f} | {vit_scratch_val_accuracy * 100:.2f}% |
| ViT (pretrained, this notebook) | Transformer, pretrained | {vit_pretrained_val_loss:.4f} | {vit_pretrained_val_accuracy * 100:.2f}% |

**Result:** Compare the two "from scratch" rows to the four pretrained rows above. If the pattern
holds, the from-scratch ViT should land near (or below) AlexNet and VGG16, while the pretrained ViT
should land in the same range as MobileNetV2 and EfficientNetB0 -- direct evidence for this
notebook's central lesson about pretraining and data efficiency. (GoogLeNet and ResNet50 aren't
listed here since their results depend on your own run of those notebooks.)
"""

display(Markdown(comparison_text))